In [7]:
!pip install imbalance-metrics smogn

In [8]:
!pip install ImbalancedLearningRegression


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import PchipInterpolator
from smogn import phi, phi_ctrl_pts
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from scipy.spatial import distance
from scipy.spatial import distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
    BaggingRegressor
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.linear_model import (
    ARDRegression,
    SGDRegressor,
    PassiveAggressiveRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from smogn import smoter
import ImbalancedLearningRegression as iblr

In [10]:
# models_pool = [
#     SVR(),
#      MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
#      XGBRegressor(n_estimators=300, learning_rate=0.05),
#     RandomForestRegressor(n_estimators=300)
# ]
models_pool = [
    SVR(),
    RandomForestRegressor(n_estimators=100, random_state=42),
    DecisionTreeRegressor(random_state=42),
    MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    BaggingRegressor(random_state=42),
    XGBRegressor(n_estimators=100, random_state=42)
]
# models_pool = [
#     RandomForestRegressor(n_estimators=100, random_state=42),
#     ExtraTreesRegressor(n_estimators=100, random_state=42),
#     XGBRegressor(n_estimators=100, random_state=42)
# ]

In [11]:
def instance_hardness_regression(X, y, models=models_pool, distance='l1', n_splits=10, logs=False, gamma_fixo = False, fator_divisao_gamma = 1):
    n = len(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    if gamma_fixo:
      gamma = 1
    else:
      ### Justamente o que diz no artigo, que é a media do target elevado a 2
      gamma = np.mean(y**2) / fator_divisao_gamma #### Gamma menor -> Maior sensibilidade a erros, deixando o valor final de IG mais proximo de 1 ( Por padrão é np.mean(y**2))

    preds_pool = np.zeros((n, len(models)))

    ### Cross-validation ###
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        if logs:
          print(f"Iniciando fold {fold_idx}/{n_splits}...")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = y[train_idx]

        ### ADICIONAR OS TIPOS DE BALANCEAMENTO NO X_TRAIN AQUI
        for j, model in enumerate(models):
            if logs:
              print(f"Treinando modelo {j+1}/{len(models)}: {type(model).__name__} ...")
            clone_model = type(model)(**model.get_params())
            clone_model.fit(X_train, y_train)
            preds_pool[test_idx, j] = clone_model.predict(X_test)

    if distance == 'l1':
        dists = np.abs(y.reshape(-1, 1) - preds_pool)
    elif distance == 'l2':
        dists = (y.reshape(-1, 1) - preds_pool)**2
    else:
        raise ValueError("Use 'l1' ou 'l2'.")

    exp_term = np.exp(-dists / gamma)
    ih_values = 1 - np.mean(exp_term, axis=1) ### Isso equivale a divisão por |L|

    return ih_values

In [12]:
## load dependencies - third party
import numpy as np
import pandas as pd
import random as rd
from tqdm import tqdm


## generate synthetic observations
def over_sampling_ro(

    ## arguments / inputs
    data,       ## training set
    index,      ## index of input data
    perc,       ## over / under sampling
    replace,    ## sampling replacement (bool)

    ):

    """
    generates synthetic observations and is the primary function underlying the
    over-sampling technique utilized in the higher main function 'ro()', the
    4 step procedure for generating synthetic observations is:

    1) pre-processing: temporarily removes features without variation, label
    encodes nominal / categorical features, and subsets the training set into
    two data sets by data type: numeric / continuous, and nominal / categorical

    2) over-sampling: RO, which random choose samples from the original samples

    3) post processing: restores original values for label encoded features,
    reintroduces constant features previously removed, converts any interpolated
    negative values to zero in the case of non-negative features

    returns a pandas dataframe containing synthetic observations of the training
    set which are then returned to the higher main function 'ro()'

    ref:

    Branco, P., Torgo, L., Ribeiro, R. (2017).
    SMOGN: A Pre-Processing Approach for Imbalanced Regression.
    Proceedings of Machine Learning Research, 74:36-50.
    http://proceedings.mlr.press/v74/branco17a/branco17a.pdf.

    Branco, P., Ribeiro, R., Torgo, L. (2017).
    Package 'UBL'. The Comprehensive R Archive Network (CRAN).
    https://cran.r-project.org/web/packages/UBL/UBL.pdf.

    Branco, P., Torgo, L., & Ribeiro, R. P. (2019).
    Pre-processing approaches for imbalanced distributions in regression.
    Neurocomputing, 343, 76-99.
    https://www.sciencedirect.com/science/article/abs/pii/S0925231219301638

    Kunz, N., (2019). SMOGN.
    https://github.com/nickkunz/smogn
    """

    ## subset original dataframe by bump classification index
    data = data.iloc[index]

    ## store dimensions of data subset
    n = len(data)
    d = len(data.columns)

    ## store original data types
    feat_dtypes_orig = [None] * d

    for j in range(d):
        feat_dtypes_orig[j] = data.iloc[:, j].dtype

    ## find non-negative numeric features
    feat_non_neg = []
    num_dtypes = ["int64", "float64"]

    for j in range(d):
        if data.iloc[:, j].dtype in num_dtypes and any(data.iloc[:, j] > 0):
            feat_non_neg.append(j)

    ## find features without variation (constant features)
    feat_const = data.columns[data.nunique() == 1]

    ## temporarily remove constant features
    if len(feat_const) > 0:

        ## create copy of orignal data and omit constant features
        data_orig = data.copy()
        data = data.drop(data.columns[feat_const], axis = 1)

        ## store list of features with variation
        feat_var = list(data.columns.values)

        ## reindex features with variation
        for i in range(d - len(feat_const)):
            data.rename(columns = {
                data.columns[i]: i
                }, inplace = True)

        ## store new dimension of feature space
        d = len(data.columns)

    ## create copy of data containing variation
    data_var = data.copy()

    ## create global feature list by column index
    feat_list = list(data.columns.values)

    ## create nominal feature list and
    ## label encode nominal / categorical features
    ## (strictly label encode, not one hot encode)
    feat_list_nom = []
    nom_dtypes = ["object", "bool", "datetime64"]

    # Unknown warning, may be handled later
    pd.options.mode.chained_assignment = None

    for j in range(d):
        if data.dtypes[j] in nom_dtypes:
            feat_list_nom.append(j)
            data.iloc[:, j] = pd.Categorical(pd.factorize(
                data.iloc[:, j])[0])

    data = data.apply(pd.to_numeric)

    ## create numeric feature list
    feat_list_num = list(set(feat_list) - set(feat_list_nom))

    ## calculate ranges for numeric / continuous features
    ## (includes label encoded features)
    feat_ranges = list(np.repeat(1, d))

    if len(feat_list_nom) > 0:
        for j in feat_list_num:
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])
    else:
        for j in range(d):
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])

    ## subset feature ranges to include only numeric features
    ## (excludes label encoded features)
    feat_ranges_num = [feat_ranges[i] for i in feat_list_num]

    ## subset data by either numeric / continuous or nominal / categorical
    data_num = data.iloc[:, feat_list_num]
    data_nom = data.iloc[:, feat_list_nom]

    ## get number of features for each data type
    feat_count_num = len(feat_list_num)
    feat_count_nom = len(feat_list_nom)


    ## number of new synthetic observations for each rare observation
    x_synth = int(perc - 1)

    ## total number of new synthetic observations to generate
    n_synth = int(n * (perc - 1 - x_synth))

    ## randomly index data by the number of new synthetic observations
    r_index = np.random.choice(
        a = tuple(range(0, n)),
        size = x_synth * n + n_synth if replace else n_synth,
        replace = replace,
        p = None
    )

    ## create null matrix to store new synthetic observations
    synth_matrix = np.ndarray(shape = ((x_synth * n + n_synth), d))

    # added
    ## store data in the synthetic matrix, data indices are chosen randomly above
    count = 0
    for i in tqdm(r_index, ascii = True, desc = "r_index"):
        for attr in range(d):
            synth_matrix[count, attr] = (data.iloc[i, attr])
        count = count + 1
    ## if the number of random chosen samples is greater than the number of samples，
    ## and replace = False,
    ## simply duplicate x_synth times the original samples
    if not replace:
        for i in tqdm(range(x_synth * n), ascii = True, desc = "duplicating_the_same_samples"):
            for attr in range(d):
                synth_matrix[count, attr] = (data.iloc[i % n, attr])
            count = count + 1


    ## convert synthetic matrix to dataframe
    data_new = pd.DataFrame(synth_matrix)

    ## synthetic data quality check
    if sum(data_new.isnull().sum()) > 0:
        raise ValueError("oops! synthetic data contains missing values")

    ## replace label encoded values with original values
    for j in feat_list_nom:
        code_list = data.iloc[:, j].unique()
        cat_list = data_var.iloc[:, j].unique()

        for x in code_list:
            data_new.iloc[:, j] = data_new.iloc[:, j].replace(x, cat_list[x])

    ## reintroduce constant features previously removed
    if len(feat_const) > 0:
        data_new.columns = feat_var

        for j in range(len(feat_const)):
            data_new.insert(
                loc = int(feat_const[j]),
                column = feat_const[j],
                value = np.repeat(
                    data_orig.iloc[0, feat_const[j]],
                    len(synth_matrix))
            )

    ## convert negative values to zero in non-negative features
    for j in feat_non_neg:
        # data_new.iloc[:, j][data_new.iloc[:, j] < 0] = 0
        data_new.iloc[:, j] = data_new.iloc[:, j].clip(lower = 0)

    ## return over-sampling results dataframe
    return data_new

In [13]:
def apply_ro_ih(
    data,
    y,
    samp_method="balance",
    drop_na_col=True,
    drop_na_row=True,
    replace=True,
    manual_perc=False,
    perc_o=-1,
    rel_thres=0.5,
    rel_method="auto",
    rel_xtrm_type="both",
    rel_coef=1.5,
    rel_ctrl_pts_rg=None
):
    """
    Versão do ro() mantendo todo o fluxo original, mas substituindo a etapa
    de phi(y) por Instance Hardness (IH) calculado pela função
    instance_hardness_regression. Basta passar `ih_models` (lista de modelos).
    """

    ## PRE-PROCESSAMENTO (IGUAL AO ORIGINAL)
    if bool(drop_na_col):
        data = data.dropna(axis=1)

    if bool(drop_na_row):
        data = data.dropna(axis=0)

    if data.isnull().values.any():
        raise ValueError("cannot proceed: data cannot contain NaN values")

    if not isinstance(y, str):
        raise ValueError("y must be a string")

    if y not in data.columns:
        raise ValueError("y must be a header in dataframe")

    if samp_method not in ["balance", "extreme"]:
        raise ValueError("samp_method must be 'balance' or 'extreme'")

    if manual_perc:
        if perc_o == -1:
            raise ValueError("require percentage of over-sampling")
        if perc_o <= 0:
            raise ValueError("percentage must be positive real")

    if rel_thres <= 0 or rel_thres > 1:
        raise ValueError("rel_thres must be 0 < R < 1")
    n = len(data)
    d = len(data.columns)

    feat_dtypes_orig = [data.iloc[:, j].dtype for j in range(d)]

    # SALVAR cópia do índice original (usado para alinhar IH)
    original_index = data.index.copy()

    # antes de mover y, extraio X e y originais para calcular IH
    X_for_ih = data.drop(columns=[y]).copy()
    y_for_ih = data[y].copy()

    # determinar coluna y e mover para a última posição (mantendo fluxo original)
    y_col = data.columns.get_loc(y)
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data = data[data.columns[cols]]

    # continuar com a codificação de colunas por índice
    feat_names = list(data.columns)
    data.columns = range(d)

    # criar y_sorted (séria com índices do dataframe original)
    y_df = pd.DataFrame(data[d - 1])
    y_sort = y_df.sort_values(by=d - 1)
    y_sort_series = y_sort[d - 1]  # Series ordenada com índices originais

    # -----------------------------
    # calcular IH usando a função fornecida (no dataset original)
    # -----------------------------
    # transformar X_for_ih e y_for_ih para numpy
    X_np = X_for_ih.to_numpy()
    y_np = y_for_ih.to_numpy().reshape(-1)

    ih_original = instance_hardness_regression(
        X_np,
        y_np
    )

    # criar Series de IH indexada pelo índice original do dataframe
    ih_series = pd.Series(ih_original, index=original_index)

    # reordenar ih para acompanhar y_sort (usando os índices ordenados de y_sort)
    y_ih = ih_series.loc[y_sort_series.index].values

    # validações análogas às do phi
    if all(i == 0 for i in y_ih):
        raise ValueError("redefine IH relevance function: all points are 0")

    if all(i == 1 for i in y_ih):
        raise ValueError("redefine IH relevance function: all points are 1")

    # ---------------------------------------------------------
    # determinar os bumps (raros vs normais) usando top 20% IH
    # ---------------------------------------------------------
    bumps = [0]
    # threshold dinâmico: percentil 80 (top 20% IH)
    dynamic_thres = np.percentile(y_ih, 70)

    for i in range(len(y_ih) - 1):
        if ((y_ih[i] >= dynamic_thres and y_ih[i + 1] < dynamic_thres) or
            (y_ih[i] < dynamic_thres and y_ih[i + 1] >= dynamic_thres)):
            bumps.append(i + 1)

    bumps.append(n)

    n_bumps = len(bumps) - 1

    b_index = {}
    for i in range(n_bumps):
        # aqui usamos y_sort (Series) com slicing para preservar índices originais,
        # como no seu código original.
        b_index.update({i: y_sort_series[bumps[i]:bumps[i + 1]]})

    # cálculo de s_perc idêntico ao original
    b = round(n / n_bumps)
    s_perc = []
    scale = []
    obj = []

    if samp_method == "balance":
        for i in b_index:
            s_perc.append(b / len(b_index[i]))

    if samp_method == "extreme":
        for i in b_index:
            scale.append(b ** 2 / len(b_index[i]))
        scale = n_bumps * b / sum(scale)

        for i in b_index:
            obj.append(round(b ** 2 / len(b_index[i]) * scale, 2))
            s_perc.append(round(obj[i] / len(b_index[i]), 1))

    data_new = pd.DataFrame()

    for i in range(n_bumps):

        if s_perc[i] == 1:
            # b_index[i] é uma Series (y values) com índices originais
            data_new = pd.concat(
                [data.iloc[b_index[i].index], data_new], ignore_index=True
            )

        if s_perc[i] > 1:

            synth_obs = over_sampling_ro(
                data=data,
                index=list(b_index[i].index),
                perc=s_perc[i] if not manual_perc else perc_o + 1,
                replace=replace
            )

            data_new = pd.concat([synth_obs, data_new], ignore_index=True)

            original_obs = data.iloc[list(b_index[i].index)]
            data_new = pd.concat([original_obs, data_new], ignore_index=True)

        if s_perc[i] < 1:
            original_obs = data.iloc[list(b_index[i].index)]
            data_new = pd.concat([original_obs, data_new], ignore_index=True)

    # restaurar nomes das colunas originais
    data_new.columns = feat_names

    # restaurar posição original de y, se necessário
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data_new = data_new[data_new.columns[cols]]

    # restaurar tipos originais
    for j in range(d):
        data_new.iloc[:, j] = data_new.iloc[:, j].astype(feat_dtypes_orig[j])

    return data_new

In [ ]:
# def hardness_bin_oversampling(df, target_col, threshold=0.8, bins=60):
#     """
#     Aplica RO apenas nos bins onde a média do IH está entre os top 20%.
#     """
#     y = df[target_col].values
#     X = df.drop(columns=[target_col]).values

#     # Instance Hardness
#     ih_scores = instance_hardness_regression(X, y)

#     # Threshold global (top 20% mais difíceis)
#     global_threshold = np.quantile(ih_scores, threshold)

#     # Gerar bins
#     counts, bin_edges = np.histogram(y, bins=bins)

#     new_df = df.copy()

#     for i in range(len(bin_edges)-1):
#         left = bin_edges[i]
#         right = bin_edges[i+1]

#         mask = (y >= left) & (y < right)
#         bin_data = df[mask]

#         if len(bin_data) == 0:
#             continue

#         ih_bin = ih_scores[mask]
#         mean_ih = np.mean(ih_bin)

#         # Se bin estiver entre os mais difíceis → aplica oversampling
#         if mean_ih >= global_threshold and len(bin_data) > 5:
#             sampled = bin_data.sample(
#                 n=len(bin_data),
#                 replace=True,
#                 random_state=42
#             )
#             new_df = pd.concat([new_df, sampled], ignore_index=True)

#     return new_df

In [14]:
def apply_iblr_ro(data, target, method="balance", rel_thres=0.8):
    """
    Wrapper do IBLR-RO

    data: DataFrame completo (X + y)
    target: nome da coluna target
    method: método de amostragem (ex: "balance", "random"...)
    rel_thres: limiar de relevância
    """
    try:
        data = data.copy()
        data.columns = data.columns.astype(str)  # garante compatibilidade

        new_data = iblr.ro(
            data=data,
            y=str(target),
            samp_method=method,
            rel_thres=rel_thres
        )
        return new_data

    except Exception as e:
        print("⚠️ IBLR-RO falhou:", e)
        return data.copy()


def apply_iblr_gn(data, target, method="balance", pert=0.1, rel_thres=0.8):
    """
    Wrapper do IBLR-GN

    data: DataFrame completo (X + y)
    target: nome da coluna target
    method: método de amostragem (ex: "balance", "random"...)
    pert: nível de perturbação
    rel_thres: limiar de relevância
    """
    try:
        data = data.copy()
        data.columns = data.columns.astype(str)  # garante compatibilidade

        new_data = iblr.gn(
            data=data,
            y=str(target),
            samp_method=method,
            pert=pert,
            rel_thres=rel_thres
        )
        return new_data

    except Exception as e:
        print("⚠️ IBLR-GN falhou:", e)
        return data.copy()



In [15]:

def evaluate_model(X_train, X_test, y_train, y_test, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # MSE
    mse = mean_squared_error(y_test, y_pred)

    # mae
    mae = mean_absolute_error(y_test, y_pred)

    return mse, mae


In [16]:
models = {
    "SVR": SVR(),  # não possui random_state
    "NNET":MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    "XGB":XGBRegressor(n_estimators=100, random_state=42),
    "RF":  RandomForestRegressor(n_estimators=100, random_state=42),
    "BAGGING": BaggingRegressor(random_state=42)
}

In [ ]:
wine = pd.read_csv('/content/drive/MyDrive/TCC/dados/wine_quality.csv')
a1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a1.csv')
a2 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a2.csv')
a3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a3.csv')
a7 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a7.csv')
acceleration = pd.read_csv('/content/drive/MyDrive/TCC/dados/acceleration.csv')
airfoild = pd.read_csv('/content/drive/MyDrive/TCC/dados/airfoild.csv')
boston = pd.read_csv('/content/drive/MyDrive/TCC/dados/boston.csv')
concreteStrength = pd.read_csv('/content/drive/MyDrive/TCC/dados/concreteStrength.csv')
heat = pd.read_csv('/content/drive/MyDrive/TCC/dados/heat.csv')
sensory = pd.read_csv('/content/drive/MyDrive/TCC/dados/sensory.csv')
analcatdata_apnea3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/analcatdata_apnea3.csv')
available_power = pd.read_csv('/content/drive/MyDrive/TCC/dados/available_power.csv')
cocomo_numeric = pd.read_csv('/content/drive/MyDrive/TCC/dados/cocomo_numeric.csv')
debutanizer = pd.read_csv('/content/drive/MyDrive/TCC/dados/debutanizer.csv')
fuel_consumption_country = pd.read_csv('/content/drive/MyDrive/TCC/dados/fuel_consumption_country.csv')

abalone = pd.read_csv('/content/drive/MyDrive/TCC/dados/abalone.csv')
california = pd.read_csv('/content/drive/MyDrive/TCC/dados/california.csv')
compactiv = pd.read_csv('/content/drive/MyDrive/TCC/dados/compactiv.csv')
cpu_small = pd.read_csv('/content/drive/MyDrive/TCC/dados/cpu_small.csv')
forestFires = pd.read_csv('/content/drive/MyDrive/TCC/dados/forestFires.csv')
kdd_coil_1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/kdd_coil_1.csv')
lungcancer_shedden = pd.read_csv('/content/drive/MyDrive/TCC/dados/lungcancer_shedden.csv')
maximal_torque = pd.read_csv('/content/drive/MyDrive/TCC/dados/maximal_torque.csv')
meta = pd.read_csv('/content/drive/MyDrive/TCC/dados/meta.csv')
mortgage = pd.read_csv('/content/drive/MyDrive/TCC/dados/mortgage.csv')
pdgfr = pd.read_csv('/content/drive/MyDrive/TCC/dados/pdgfr.csv')
space_ga = pd.read_csv('/content/drive/MyDrive/TCC/dados/space_ga.csv')
treasury = pd.read_csv('/content/drive/MyDrive/TCC/dados/treasury.csv')
triazines = pd.read_csv('/content/drive/MyDrive/TCC/dados/triazines.csv')



datasets = {
    "a1": {"data": a1, "target": "a1"},
    "a2": {"data": a2, "target": "a2"},
    "a3": {"data": a3, "target": "a3"},
    "a7": {"data": a7, "target": "a7"},
     "abalone": {"data": abalone, "target": "Rings"},
     "acceleration": {"data": acceleration, "target": "target"},
    "airfoild": {"data": airfoild, "target": "target"},
    "analcatdata_apnea3": {"data": analcatdata_apnea3, "target": "target"},
     "available_power": {"data": available_power, "target": "target"},
   "boston": {"data": boston, "target": "target"},
    #"california": {"data": california, "target": "target"},
    "cocomo_numeric": {"data": cocomo_numeric, "target": "target"},
    "compactiv": {"data": compactiv, "target": "target"},
     "concreteStrength": {"data": concreteStrength, "target": "target"},
     "cpu_small": {"data": cpu_small, "target": "usr"},
     "debutanizer": {"data": debutanizer, "target": "target"},
    "fuel_consumption_country": {"data": fuel_consumption_country, "target": "fuel.consumption.country"},
    "forestFires": {"data": forestFires, "target": "Area"},
     "heat": {"data": heat, "target": "heat"},
    "kdd_coil_1": {"data": kdd_coil_1, "target": "target"},
    "lungcancer_shedden": {"data": lungcancer_shedden, "target": "target"},
     "maximal_torque": {"data": maximal_torque, "target": "maximal.torque"},
    "meta": {"data": meta, "target": "target"},
    "mortgage": {"data": mortgage, "target": "target"},
    "pdgfr": {"data": pdgfr, "target": "target"},
    "sensory": {"data": sensory, "target": "target"},
     "space_ga": {"data": space_ga, "target": "target"},
    "treasury": {"data": treasury, "target": "target"},
    "triazines": {"data": triazines, "target": "target"},
    "wine": {"data": wine, "target": "target"},
}


Teste para RO

In [ ]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

results = []

for name, info in datasets.items():
    df = info["data"].copy()
    target = info["target"]

    print(f"\nProcessando: {name}")

    X = df.drop(columns=[target])
    y = df[target]

    # Loop RepeatedKFold
    for fold_id, (train_idx, test_idx) in enumerate(rkf.split(X)):
        repeat = fold_id // 5 + 1
        fold = fold_id % 5 + 1

        print(f"  Repetição {repeat}/2 - Fold {fold}/5")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        base_train = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)

        # Oversampling dentro do fold
        try:
            train_smogn = apply_iblr_ro(base_train, target)
        except Exception as e:
            print(f"⚠️ RO falhou no fold {fold} da repetição {repeat}: {e}")
            train_smogn = base_train.copy()

        train_hard = apply_ro_ih(base_train, target)

        versions = {
            "Original": base_train,
            "RO": train_smogn,
            "Hardness-RO": train_hard
        }

        for model_name, model in models.items():
            for version_name, train_data in versions.items():

                Xtr = train_data.drop(columns=[target])
                ytr = train_data[target]

                Xts = X_test
                yts = y_test

                # Ajuste e predição
                model.fit(Xtr, ytr)
                y_pred = model.predict(Xts)

                # Cálculo das métricas
                mse = mean_squared_error(yts, y_pred)
                mae = mean_absolute_error(yts, y_pred)

                results.append({
                    "Dataset": name,
                    "Repeticao": repeat,
                    "Fold": fold,
                    "Model": model_name,
                    "Versao": version_name,
                    "MSE": mse,
                    "MAE": mae,
                    "y_pred": y_pred.tolist(),   # <<<<<<<< AQUI EXPORTA O Y PREDITO
                    "y_true": yts.tolist()        # (opcional mas recomendado)
                })


NameError: name 'datasets' is not defined

In [ ]:
results_df = pd.DataFrame(results)
results_df.head(50)


In [ ]:
results_df.to_csv("comparacao_final_RO.csv", index=False)

In [18]:
results_df = pd.read_csv('/content/drive/MyDrive/TCC/comparacao_final_RO_07_repeat.csv')

In [19]:
# Definir a ordem desejada
ordem_versoes = ["Original", "RO", "Hardness-RO"]

results_df["Versao"] = pd.Categorical(
    results_df["Versao"],
    categories=ordem_versoes,
    ordered=True
)

mse_por_modelo = (
    results_df
    .groupby(["Dataset", "Model", "Versao"], as_index=False)
    .agg(
        MSE_medio=("MSE", "mean"),
        MAE_medio=("MAE", "mean")
        )
    .sort_values(["Dataset", "Model", "Versao"])
)

mse_por_modelo.head(60)

/tmp/ipython-input-4070163773.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Dataset", "Model", "Versao"], as_index=False)


,Dataset,Model,Versao,MSE_medio,MAE_medio
0,a1,BAGGING,Original,3.217092e+02,12.160385
1,a1,BAGGING,RO,3.237340e+02,12.701449
2,a1,BAGGING,Hardness-RO,3.126563e+02,12.314399
3,a1,NNET,Original,1.015772e+06,282.608819
4,a1,NNET,RO,1.099930e+06,384.176189
5,a1,NNET,Hardness-RO,1.118966e+06,317.442029
6,a1,RF,Original,2.776386e+02,11.515071
7,a1,RF,RO,2.970657e+02,11.972114
8,a1,RF,Hardness-RO,2.818767e+02,11.699547
9,a1,SVR,Original,4.783005e+02,14.380785


### Vendo quem foi melhor pro MSE

In [20]:
import pandas as pd

# ===============================
# 1. Ordenar versões corretamente
# ===============================

ordem_versoes = ["Original", "RO", "Hardness-RO"]

results_df["Versao"] = pd.Categorical(
    results_df["Versao"],
    categories=ordem_versoes,
    ordered=True
)

# ===============================
# 2. Agregar MSE médio por fold
# ===============================

mse_por_modelo = (
    results_df
    .groupby(["Dataset", "Model", "Versao"], as_index=False)
    .agg(MSE_medio=("MSE", "mean"))
    .sort_values(["Dataset", "Model", "Versao"])
)

print("\n📊 MSE médio por versão:")
print(mse_por_modelo)

# ===============================
# 3. Comparação entre RO e Hardness
# ===============================

# Filtrar somente RO e Hardness-RO
comp = mse_por_modelo[
    mse_por_modelo["Versao"].isin(["RO", "Hardness-RO"])
]

# Pivot: transformar em colunas RO e Hardness-RO
pivot = comp.pivot(
    index=["Dataset", "Model"],
    columns="Versao",
    values="MSE_medio"
).reset_index()

print("\n📌 Comparação direta:")
print(pivot)

# ===============================
# 4. Quem foi melhor?
# ===============================

pivot["Hardness_melhor_que_RO"] = pivot["Hardness-RO"] < pivot["RO"]
pivot["RO_melhor_que_Hardness"] = pivot["RO"] < pivot["Hardness-RO"]
pivot["Empate"] = pivot["RO"] == pivot["Hardness-RO"]

# ===============================
# 5. Contagem geral
# ===============================

total_hardness = pivot["Hardness_melhor_que_RO"].sum()
total_ro = pivot["RO_melhor_que_Hardness"].sum()
empates = pivot["Empate"].sum()

print(f"\n🏆 RESULTADO GERAL:")
print(f"Hardness-RO foi melhor em {total_hardness} pares (dataset, modelo)")
print(f"RO foi melhor em {total_ro} pares (dataset, modelo)")
print(f"Houve {empates} empates")

# ===============================
# 6. Contagem por dataset
# ===============================

hardness_por_dataset = (
    pivot[pivot["Hardness_melhor_que_RO"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_Hardness")
)

ro_por_dataset = (
    pivot[pivot["RO_melhor_que_Hardness"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_RO")
)

print("\n📌 Hardness-RO melhor que RO por dataset:")
print(hardness_por_dataset)

print("\n📌 RO melhor que Hardness-RO por dataset:")
print(ro_por_dataset)

# ===============================
# 7. Frases finais (estilo artigo)
# ===============================

datasets_hardness = hardness_por_dataset["Dataset"].nunique()
datasets_ro = ro_por_dataset["Dataset"].nunique()

modelos_hardness = hardness_por_dataset["Modelos_ganhos_Hardness"]



📊 MSE médio por versão:
    Dataset    Model       Versao     MSE_medio
0        a1  BAGGING     Original  3.217092e+02
1        a1  BAGGING           RO  3.237340e+02
2        a1  BAGGING  Hardness-RO  3.126563e+02
3        a1     NNET     Original  1.015772e+06
4        a1     NNET           RO  1.099930e+06
..      ...      ...          ...           ...
430    wine      SVR           RO  7.051622e-01
431    wine      SVR  Hardness-RO  6.363423e-01
432    wine      XGB     Original  4.047501e-01
433    wine      XGB           RO  4.352097e-01
434    wine      XGB  Hardness-RO  4.073976e-01

[435 rows x 4 columns]

📌 Comparação direta:
Versao Dataset    Model            RO   Hardness-RO
0           a1  BAGGING  3.237340e+02  3.126563e+02
1           a1     NNET  1.099930e+06  1.118966e+06
2           a1       RF  2.970657e+02  2.818767e+02
3           a1      SVR  4.125509e+02  4.851627e+02
4           a1      XGB  3.356241e+02  3.513928e+02
..         ...      ...           ...    

/tmp/ipython-input-2671954749.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Dataset", "Model", "Versao"], as_index=False)


In [21]:
# Separar RO e Hardness-RO
ro_df = mse_por_modelo[mse_por_modelo["Versao"] == "RO"]
hard_df = mse_por_modelo[mse_por_modelo["Versao"] == "Hardness-RO"]

# Juntar lado a lado
comparacao = pd.merge(
    ro_df,
    hard_df,
    on=["Dataset", "Model"],
    suffixes=("_RO", "_HardnessRO")
)

# Criar colunas indicando quem foi melhor (menor MSE)
comparacao["Hardness_melhor"] = comparacao["MSE_medio_HardnessRO"] < comparacao["MSE_medio_RO"]
comparacao["RO_melhor"] = comparacao["MSE_medio_RO"] < comparacao["MSE_medio_HardnessRO"]
comparacao["Empate"] = comparacao["MSE_medio_RO"] == comparacao["MSE_medio_HardnessRO"]

datasets_ro_ganhou = (
    comparacao[comparacao["RO_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_RO_ganhou")
)

datasets_ro_ganhou



,Dataset,Modelos_que_RO_ganhou
0,a1,"[NNET, SVR, XGB]"
1,a2,[SVR]
2,a3,[SVR]
3,a7,"[BAGGING, NNET, SVR]"
4,airfoild,"[BAGGING, RF, SVR, XGB]"
5,analcatdata_apnea3,"[NNET, SVR]"
6,available_power,"[BAGGING, NNET, SVR]"
7,boston,"[BAGGING, RF, XGB]"
8,cocomo_numeric,"[BAGGING, SVR, XGB]"
9,compactiv,"[BAGGING, XGB]"


In [22]:
datasets_hard_ganhou = (
    comparacao[comparacao["Hardness_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_HardnessRO_ganhou")
)

datasets_hard_ganhou


,Dataset,Modelos_que_HardnessRO_ganhou
0,a1,"[BAGGING, RF]"
1,a2,"[BAGGING, NNET, RF, XGB]"
2,a3,"[BAGGING, NNET, RF, XGB]"
3,a7,"[RF, XGB]"
4,abalone,"[BAGGING, NNET, RF, SVR, XGB]"
5,acceleration,"[BAGGING, NNET, RF, SVR, XGB]"
6,airfoild,[NNET]
7,analcatdata_apnea3,"[BAGGING, RF, XGB]"
8,available_power,"[RF, XGB]"
9,boston,"[NNET, SVR]"


In [23]:
# Agrupar por modelo e contar vitórias
comparacao_por_modelo = (
    comparacao
    .groupby("Model")
    .agg(
        Vitorias_RO=("RO_melhor", "sum"),
        Vitorias_HardnessRO=("Hardness_melhor", "sum"),
        Empates=("Empate", "sum"),
        Total_Comparacoes=("Model", "count")
    )
    .reset_index()
)

# Criar também a diferença de vitórias
comparacao_por_modelo["Dif_Hardness_minus_RO"] = (
    comparacao_por_modelo["Vitorias_HardnessRO"]
    - comparacao_por_modelo["Vitorias_RO"]
)

comparacao_por_modelo.sort_values("Dif_Hardness_minus_RO", ascending=False)


,Model,Vitorias_RO,Vitorias_HardnessRO,Empates,Total_Comparacoes,Dif_Hardness_minus_RO
2,RF,5,24,0,29,19
4,XGB,9,20,0,29,11
0,BAGGING,10,19,0,29,9
3,SVR,12,17,0,29,5
1,NNET,13,16,0,29,3


### Vendo quem foi melhor pro mae

In [24]:
import pandas as pd

# ===============================
# 1. Ordenar versões corretamente
# ===============================

ordem_versoes = ["Original", "RO", "Hardness-RO"]

results_df["Versao"] = pd.Categorical(
    results_df["Versao"],
    categories=ordem_versoes,
    ordered=True
)

# ===============================
# 2. Agregar mae médio por fold
# ===============================

mae_por_modelo = (
    results_df
    .groupby(["Dataset", "Model", "Versao"], as_index=False)
    .agg(mae_medio=("MAE", "mean"))
    .sort_values(["Dataset", "Model", "Versao"])
)

print("\n📊 mae médio por versão:")
print(mae_por_modelo)

# ===============================
# 3. Comparação entre RO e Hardness
# ===============================

# Filtrar apenas RO e Hardness-RO
comp = mae_por_modelo[
    mae_por_modelo["Versao"].isin(["RO", "Hardness-RO"])
]

# Pivot para comparar lado a lado
pivot = comp.pivot(
    index=["Dataset", "Model"],
    columns="Versao",
    values="mae_medio"
).reset_index()

print("\n📌 Comparação direta (mae):")
print(pivot)

# ===============================
# 4. Quem foi melhor?
# ===============================

pivot["Hardness_melhor_que_RO"] = pivot["Hardness-RO"] < pivot["RO"]
pivot["RO_melhor_que_Hardness"] = pivot["RO"] < pivot["Hardness-RO"]
pivot["Empate"] = pivot["RO"] == pivot["Hardness-RO"]

# ===============================
# 5. Contagem geral
# ===============================

total_hardness = pivot["Hardness_melhor_que_RO"].sum()
total_ro = pivot["RO_melhor_que_Hardness"].sum()
empates = pivot["Empate"].sum()

print(f"\n🏆 RESULTADO GERAL (mae):")
print(f"Hardness-RO foi melhor em {total_hardness} pares (dataset, modelo)")
print(f"RO foi melhor em {total_ro} pares (dataset, modelo)")
print(f"Houve {empates} empates")

# ===============================
# 6. Contagem por dataset
# ===============================

hardness_por_dataset = (
    pivot[pivot["Hardness_melhor_que_RO"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_Hardness")
)

ro_por_dataset = (
    pivot[pivot["RO_melhor_que_Hardness"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_RO")
)

print("\n📌 Hardness-RO melhor que RO por dataset (mae):")
print(hardness_por_dataset)

print("\n📌 RO melhor que Hardness-RO por dataset (mae):")
print(ro_por_dataset)

# ===============================
# 7. Frases finais (estilo artigo)
# ===============================

datasets_hardness = hardness_por_dataset["Dataset"].nunique()
datasets_ro = ro_por_dataset["Dataset"].nunique()

modelos_hardness = hardness_por_dataset["Modelos_ganhos_Hardness"].sum()
modelos_ro = ro_por_dataset["Modelos_ganhos_RO"].sum()

print("\n📄 Resumo para artigo (mae):")
print(
    f"O método Hardness-RO apresentou melhor desempenho em {datasets_hardness} datasets, "
    f"superando RO em {modelos_hardness} combinações de modelos. "
    f"Por outro lado, o método RO obteve melhor desempenho em {datasets_ro} datasets, "
    f"vencendo em {modelos_ro} combinações."
)



📊 mae médio por versão:
    Dataset    Model       Versao   mae_medio
0        a1  BAGGING     Original   12.160385
1        a1  BAGGING           RO   12.701449
2        a1  BAGGING  Hardness-RO   12.314399
3        a1     NNET     Original  282.608819
4        a1     NNET           RO  384.176189
..      ...      ...          ...         ...
430    wine      SVR           RO    0.654870
431    wine      SVR  Hardness-RO    0.620201
432    wine      XGB     Original    0.457711
433    wine      XGB           RO    0.478511
434    wine      XGB  Hardness-RO    0.463647

[435 rows x 4 columns]

📌 Comparação direta (mae):
Versao Dataset    Model          RO  Hardness-RO
0           a1  BAGGING   12.701449    12.314399
1           a1     NNET  384.176189   317.442029
2           a1       RF   11.972114    11.699547
3           a1      SVR   16.027205    14.476596
4           a1      XGB   12.290946    12.794672
..         ...      ...         ...          ...
140       wine  BAGGING    0

/tmp/ipython-input-1476718455.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Dataset", "Model", "Versao"], as_index=False)


In [25]:
# Separar RO e Hardness-RO usando mae_medio
ro_df = mae_por_modelo[mae_por_modelo["Versao"] == "RO"]
hard_df = mae_por_modelo[mae_por_modelo["Versao"] == "Hardness-RO"]

# Juntar lado a lado
comparacao_mae = pd.merge(
    ro_df,
    hard_df,
    on=["Dataset", "Model"],
    suffixes=("_RO", "_HardnessRO")
)

# Criar colunas indicando quem foi melhor (menor mae)
comparacao_mae["Hardness_melhor"] = (
    comparacao_mae["mae_medio_HardnessRO"] < comparacao_mae["mae_medio_RO"]
)

comparacao_mae["RO_melhor"] = (
    comparacao_mae["mae_medio_RO"] < comparacao_mae["mae_medio_HardnessRO"]
)

comparacao_mae["Empate"] = (
    comparacao_mae["mae_medio_RO"] == comparacao_mae["mae_medio_HardnessRO"]
)

# Listar os modelos em que RO ganhou por dataset
datasets_ro_ganhou_phi = (
    comparacao_mae[comparacao_mae["RO_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_RO_ganhou")
)

datasets_ro_ganhou_phi


,Dataset,Modelos_que_RO_ganhou
0,a1,[XGB]
1,a2,[NNET]
2,acceleration,"[BAGGING, XGB]"
3,airfoild,"[BAGGING, RF, XGB]"
4,available_power,"[BAGGING, SVR]"
5,boston,[BAGGING]
6,cocomo_numeric,"[BAGGING, RF, XGB]"
7,compactiv,[RF]
8,concreteStrength,"[NNET, XGB]"
9,cpu_small,[BAGGING]


In [26]:
datasets_hard_ganhou_mae = (
    comparacao_mae[comparacao_mae["Hardness_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_HardnessRO_ganhou")
)

datasets_hard_ganhou_mae


,Dataset,Modelos_que_HardnessRO_ganhou
0,a1,"[BAGGING, NNET, RF, SVR]"
1,a2,"[BAGGING, RF, SVR, XGB]"
2,a3,"[BAGGING, NNET, RF, SVR, XGB]"
3,a7,"[BAGGING, NNET, RF, SVR, XGB]"
4,abalone,"[BAGGING, NNET, RF, SVR, XGB]"
5,acceleration,"[NNET, RF, SVR]"
6,airfoild,"[NNET, SVR]"
7,analcatdata_apnea3,"[BAGGING, NNET, RF, SVR, XGB]"
8,available_power,"[NNET, RF, XGB]"
9,boston,"[NNET, RF, SVR, XGB]"


In [27]:
# Agrupar por modelo e contar vitórias (mae)
comparacao_por_modelo_mae = (
    comparacao_mae
    .groupby("Model")
    .agg(
        Vitorias_RO=("RO_melhor", "sum"),
        Vitorias_HardnessRO=("Hardness_melhor", "sum"),
        Empates=("Empate", "sum"),
        Total_Comparacoes=("Model", "count")
    )
    .reset_index()
)

# Criar também a diferença de vitórias
comparacao_por_modelo_mae["Dif_Hardness_minus_RO"] = (
    comparacao_por_modelo_mae["Vitorias_HardnessRO"]
    - comparacao_por_modelo_mae["Vitorias_RO"]
)

# Ordenar pela diferença (quem mais ganhou)
comparacao_por_modelo_mae.sort_values("Dif_Hardness_minus_RO", ascending=False)


,Model,Vitorias_RO,Vitorias_HardnessRO,Empates,Total_Comparacoes,Dif_Hardness_minus_RO
3,SVR,1,28,0,29,27
2,RF,4,25,0,29,21
1,NNET,5,24,0,29,19
4,XGB,7,22,0,29,15
0,BAGGING,8,21,0,29,13


## Erro por area

In [ ]:
def plot_binned_mae(y_true, y_pred, n_bins=10, title="Binned MAE"):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    abs_error = np.abs(y_true - y_pred)

    # Criar bins
    bins = np.linspace(y_true.min(), y_true.max(), n_bins + 1)

    # Atribuir cada valor a um bin
    bin_ids = np.digitize(y_true, bins) - 1

    binned_mae = []
    bin_centers = []

    for i in range(n_bins):
        mask = bin_ids == i

        if np.sum(mask) == 0:
            binned_mae.append(np.nan)
        else:
            binned_mae.append(np.mean(abs_error[mask]))

        bin_centers.append((bins[i] + bins[i+1]) / 2)

    # Plot
    fig, ax1 = plt.subplots(figsize=(9,5))

    # Histograma
    ax1.hist(y_true, bins=bins, color='indianred', alpha=0.6)
    ax1.set_ylabel("Count", color='indianred')
    ax1.set_xlabel("Target")
    ax1.set_title(title)

    # Segundo eixo para MAE
    ax2 = ax1.twinx()
    ax2.plot(bin_centers, binned_mae, marker='o')
    ax2.set_ylabel("Binned MAE")

    plt.show()

In [ ]:

from sklearn.model_selection import train_test_split

for name, info in datasets.items():
    df = info["data"].copy()
    target = info["target"]

    print(f"\n📊 Dataset: {name}")

    X = df.drop(columns=[target])
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    train_original = pd.concat([X_train, y_train], axis=1)

    for model_name, model in models.items():
        print(f"  🔷 Modelo: {model_name} (Original)")

        Xtr = train_original.drop(columns=[target])
        ytr = train_original[target]

        model.fit(Xtr, ytr)

        y_pred = model.predict(X_test)

        plot_binned_mae(
            y_true=y_test,
            y_pred=y_pred,
            n_bins=15,
            title=f"{name} - {model_name} - Original"
        )


Output hidden; open in https://colab.research.google.com to view.